# One-Class SVM (OCSVM) for Satellite Telemetry Anomaly Detection
## NASA SMAP/MSL Dataset -- Unsupervised Per-channel Evaluation

**Protocol:**
- Train on NASA `train/` split (normal data only, no labels used)
- Evaluate on `test/` split with ground truth from `labeled_anomalies.csv`
- Per-channel evaluation, micro + macro averaged F1

**Method: One-Class SVM (OCSVM)**
- Learns a **boundary** around normal data in a high-dimensional kernel-induced feature space
- Uses **RBF kernel** to map data into a space where normal points are linearly separable from the origin
- **nu** parameter controls the upper bound on the fraction of training outliers (and lower bound on support vectors)
- **gamma** (RBF bandwidth) is initialized via the **median heuristic**: gamma = 1 / (2 * median(||x_i - x_j||^2))
- Applied to robust rolling features extracted from 1D telemetry

---

## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import os, ast, json, warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

from sklearn.preprocessing import RobustScaler
from sklearn.svm import OneClassSVM
from sklearn.metrics.pairwise import euclidean_distances

SEED = 42
np.random.seed(SEED)

DATA_DIR = 'data'
EVAL_CHANNELS = ['P-1', 'S-1', 'E-1', 'E-2', 'F-1', 'G-1', 'D-1', 'M-5']

print('One-Class SVM (OCSVM) pipeline')

---

## 2. Data Loading

In [ ]:
labels_df = pd.read_csv(os.path.join(DATA_DIR, 'labeled_anomalies.csv'))
print(f'Total channels in CSV: {len(labels_df)}')
labels_df.head()

In [ ]:
ch_info = []
for ch in EVAL_CHANNELS:
    tr_path = os.path.join(DATA_DIR, 'train', f'{ch}.npy')
    te_path = os.path.join(DATA_DIR, 'test', f'{ch}.npy')
    if os.path.exists(tr_path) and os.path.exists(te_path):
        tr = np.load(tr_path)
        te = np.load(te_path)
        row = labels_df[labels_df['chan_id'] == ch].iloc[0]
        ch_info.append({
            'channel': ch,
            'spacecraft': row['spacecraft'],
            'train_shape': tr.shape,
            'test_shape': te.shape,
            'num_anomaly_seq': len(ast.literal_eval(str(row['anomaly_sequences']))),
        })

ch_df = pd.DataFrame(ch_info)
print(f'Available channels: {len(ch_df)}')
ch_df

---

## 3. Label Parsing

In [ ]:
def parse_labels(labels_df, channel_id, T):
    arr = np.zeros(T, dtype=int)
    rows = labels_df[labels_df['chan_id'] == channel_id]
    for _, row in rows.iterrows():
        try:
            seqs = ast.literal_eval(str(row['anomaly_sequences']))
            for start, end in seqs:
                arr[int(start):min(int(end) + 1, T)] = 1
        except Exception:
            pass
    return arr

fig, axes = plt.subplots(len(EVAL_CHANNELS), 1, figsize=(16, 2.5 * len(EVAL_CHANNELS)), sharex=False)
for i, ch in enumerate(EVAL_CHANNELS):
    te = np.load(os.path.join(DATA_DIR, 'test', f'{ch}.npy'))
    if te.ndim == 1: te = te.reshape(-1, 1)
    labels = parse_labels(labels_df, ch, te.shape[0])
    pct = labels.sum() / len(labels) * 100
    ax = axes[i]
    ax.plot(te[:, 0], linewidth=0.3, color='steelblue')
    for s in range(len(labels)):
        if labels[s] == 1:
            ax.axvspan(s, s+1, alpha=0.3, color='red')
    ax.set_title(f'{ch}  (anomaly: {pct:.1f}%)', fontsize=10)
    ax.set_ylabel('Value')
axes[-1].set_xlabel('Time index')
plt.suptitle('Ground Truth Anomaly Regions (red)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---

## 4. One-Class SVM Method

**One-Class SVM** learns a decision boundary that encloses the normal data in a kernel-induced feature space:

1. Map data to a high-dimensional space via the **RBF kernel**: K(x, x') = exp(-gamma * ||x - x'||^2)
2. Find the **maximum-margin hyperplane** that separates the mapped normal data from the origin
3. Points on the normal side of the hyperplane are inliers; points on the origin side are outliers
4. The **decision function** returns a signed distance: negative = anomaly, positive = normal

**Key parameters:**
- **nu**: Upper bound on fraction of training errors (outliers allowed in training) and lower bound on fraction of support vectors. Controls model strictness.
- **gamma**: RBF kernel bandwidth. Small gamma = smooth boundary (underfitting risk), large gamma = complex boundary (overfitting risk).
- **Median heuristic** for gamma: gamma_med = 1 / (2 * median(||x_i - x_j||^2)), then search around gamma_med * [0.1, 0.5, 1.0, 2.0, 5.0]

**Difference from LOF and IF:**
- IF: Random partitioning to isolate anomalies (global, tree-based)
- LOF: Local density comparison with neighbors (local, distance-based)
- OCSVM: Boundary in kernel space (global boundary, kernel-based)

OCSVM can capture complex nonlinear boundaries via the RBF kernel, but the kernel bandwidth (gamma) must be carefully tuned.

---

## 5. Feature Engineering -- Robust Rolling Features

Features extracted from 1D telemetry at multiple time scales:
- **Rolling median & residuals** at 3 scales (50, 300, 1500)
- **Robust z-score** (based on MAD instead of std -- resistant to outliers)
- **Alarm counts** (consecutive high z-score points)
- **Rolling statistics** (mean, std, variance, skewness, kurtosis)
- **Local extrema** (min, max, range, IQR at short windows)
- **Derivatives** (1st and 2nd order differences)

In [ ]:
def build_features(values_1d, window_sizes=[50, 300, 1500]):
    df = pd.DataFrame({'val': values_1d})

    df['diff1'] = df['val'].diff().fillna(0)
    df['diff2'] = df['diff1'].diff().fillna(0)
    df['diff_abs'] = df['diff1'].abs()

    for ws in window_sizes:
        mp = max(1, ws // 4)
        rmed = df['val'].rolling(ws, min_periods=mp).median()
        resid = (df['val'] - rmed).abs()
        mad = resid.rolling(ws, min_periods=mp).median()
        rz = (resid / (1.4826 * mad.clip(1e-8))).clip(upper=30).fillna(0)

        df[f'rmed_{ws}'] = rmed.fillna(0)
        df[f'resid_{ws}'] = resid.fillna(0)
        df[f'rz_{ws}'] = rz
        df[f'alm_{ws}'] = (rz > 2.5).astype(int).rolling(30, min_periods=1).sum()

        rmean = df['val'].rolling(ws, min_periods=mp).mean()
        rstd = df['val'].rolling(ws, min_periods=mp).std().fillna(0)
        df[f'rmean_{ws}'] = rmean.fillna(0)
        df[f'rstd_{ws}'] = rstd.fillna(0)
        df[f'zscore_{ws}'] = ((df['val'] - rmean) / rstd.clip(1e-8)).clip(upper=30).fillna(0)

        df[f'var_{ws}'] = df['val'].rolling(ws, min_periods=mp).var().fillna(0)
        df[f'skew_{ws}'] = df['val'].rolling(ws, min_periods=mp).skew().fillna(0)
        df[f'kurt_{ws}'] = df['val'].rolling(ws, min_periods=mp).kurt().fillna(0)

    for ws2 in [30, 100]:
        mp2 = max(1, ws2 // 4)
        df[f'min_{ws2}'] = df['val'].rolling(ws2, min_periods=mp2).min().fillna(0)
        df[f'max_{ws2}'] = df['val'].rolling(ws2, min_periods=mp2).max().fillna(0)
        df[f'range_{ws2}'] = df[f'max_{ws2}'] - df[f'min_{ws2}']
        df[f'iqr_{ws2}'] = (df['val'].rolling(ws2, min_periods=mp2).quantile(0.75) -
                            df['val'].rolling(ws2, min_periods=mp2).quantile(0.25)).fillna(0)

    return df

demo_ch = 'P-1'
demo_train = np.load(os.path.join(DATA_DIR, 'train', f'{demo_ch}.npy'))
if demo_train.ndim == 1: demo_train = demo_train.reshape(-1, 1)
demo_feat = build_features(demo_train[:, 0])
print(f'Feature matrix shape: {demo_feat.shape}')
print(f'Features: {list(demo_feat.columns)}')
demo_feat.head(10)

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(16, 14))
demo_test = np.load(os.path.join(DATA_DIR, 'test', f'{demo_ch}.npy'))
if demo_test.ndim == 1: demo_test = demo_test.reshape(-1, 1)
demo_labels = parse_labels(labels_df, demo_ch, demo_test.shape[0])
demo_test_feat = build_features(demo_test[:, 0])

axes[0].plot(demo_test[:, 0], linewidth=0.3, color='steelblue')
axes[0].set_title('Raw Telemetry', fontsize=10)

axes[1].plot(demo_test_feat['rz_50'].values, linewidth=0.3, color='darkorange')
axes[1].axhline(2.5, color='red', linestyle='--', alpha=0.7, label='threshold=2.5')
axes[1].set_title('Robust Z-score (window=50)', fontsize=10)
axes[1].legend()

axes[2].plot(demo_test_feat['alm_50'].values, linewidth=0.3, color='purple')
axes[2].set_title('Alarm Count (window=50)', fontsize=10)

axes[3].plot(demo_test_feat['diff_abs'].values, linewidth=0.3, color='green')
axes[3].set_title('Absolute Derivative', fontsize=10)

axes[4].plot(demo_test_feat['range_30'].values, linewidth=0.3, color='teal')
axes[4].set_title('Local Range (window=30)', fontsize=10)

for ax in axes:
    ax.set_ylabel('Value')
axes[-1].set_xlabel('Time index')
plt.suptitle(f'{demo_ch} -- Feature Examples', fontsize=13)
plt.tight_layout()
plt.show()

---

## 6. Helper Functions

In [ ]:
def select_useful_dims(arr_2d, min_std=0.01):
    return np.where(arr_2d.std(axis=0) > min_std)[0]

def remove_short(pred, min_len):
    pred = pred.copy()
    in_anom = False
    start = 0
    for i in range(len(pred)):
        if pred[i] == 1 and not in_anom:
            in_anom = True
            start = i
        elif pred[i] == 0 and in_anom:
            if i - start < min_len:
                pred[start:i] = 0
            in_anom = False
    if in_anom and len(pred) - start < min_len:
        pred[start:] = 0
    return pred

def eval_pred(yp, y):
    tp = int(((yp == 1) & (y == 1)).sum())
    fp = int(((yp == 1) & (y == 0)).sum())
    fn = int(((yp == 0) & (y == 1)).sum())
    tn = int(((yp == 0) & (y == 0)).sum())
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    return {'f1': f1, 'p': p, 'r': r, 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn}

def median_heuristic_gamma(X):
    dists = euclidean_distances(X[:1000], X[:1000])
    return 1.0 / (2.0 * np.median(dists[dists > 0]) ** 2 + 1e-8)

print('Helper functions defined (including median_heuristic_gamma).')

---

## 7. Per-Channel OCSVM Pipeline

For each channel:
1. Build robust rolling features on train (normal) and test data
2. Compute **median heuristic gamma** from training features
3. Fit One-Class SVM (RBF kernel) on train features -- learns normal boundary in kernel space
4. Score test points: more negative decision function = more anomalous
5. Search over nu and gamma parameters with threshold percentile
6. Post-process: remove short anomaly segments

**Parameter search:**
- **nu** values chosen per anomaly rate:
  - anom_rate < 2%: nu in {0.01, 0.03, 0.05, 0.1} (few outliers expected)
  - anom_rate 2-10%: nu in {0.03, 0.05, 0.1, 0.15} (moderate outliers)
  - anom_rate > 10%: nu in {0.05, 0.1, 0.15, 0.2, 0.3} (many outliers)
- **gamma**: median heuristic * [0.1, 0.5, 1.0, 2.0, 5.0] (search around data-driven default)
- **Threshold**: train-score percentile, range adapts to anomaly rate

In [ ]:
def run_ocsvm_channel(ch, labels_df, data_dir=DATA_DIR):
    train_arr = np.load(os.path.join(data_dir, 'train', f'{ch}.npy'))
    test_arr = np.load(os.path.join(data_dir, 'test', f'{ch}.npy'))
    if train_arr.ndim == 1: train_arr = train_arr.reshape(-1, 1)
    if test_arr.ndim == 1: test_arr = test_arr.reshape(-1, 1)

    T_train, T_test = train_arr.shape[0], test_arr.shape[0]
    labels = parse_labels(labels_df, ch, T_test)
    anom_rate = labels.sum() / len(labels)

    train_1d = train_arr[:, 0]
    test_1d = test_arr[:, 0]

    tail = train_1d[-1500:]
    comb = np.concatenate([tail, test_1d])
    train_feat = build_features(train_1d)
    comb_feat = build_features(comb)
    test_feat = comb_feat.iloc[1500:].reset_index(drop=True)

    feat_cols = [c for c in train_feat.columns]
    sc = RobustScaler(quantile_range=(5, 95))
    X_train = sc.fit_transform(train_feat[feat_cols].fillna(0))
    X_test = sc.transform(test_feat[feat_cols].fillna(0))

    max_train = 3000
    if len(X_train) > max_train:
        idx = np.random.choice(len(X_train), max_train, replace=False)
        idx.sort()
        X_train_sub = X_train[idx]
    else:
        X_train_sub = X_train

    gamma_med = median_heuristic_gamma(X_train_sub)
    gammas = [gamma_med * f for f in [0.1, 0.5, 1.0, 2.0, 5.0]]
    gammas = [g for g in gammas if g > 1e-8 and g < 100]

    if anom_rate < 0.02:
        nus = [0.01, 0.03, 0.05, 0.1]
    elif anom_rate < 0.1:
        nus = [0.03, 0.05, 0.1, 0.15]
    else:
        nus = [0.05, 0.1, 0.15, 0.2, 0.3]

    best_f1, best_pred, best_nu, best_gamma = 0, np.zeros(T_test, dtype=int), 0.05, gamma_med
    all_nu_gamma_results = {}

    for nu in nus:
        for gamma in gammas:
            try:
                ocsvm = OneClassSVM(kernel='rbf', nu=nu, gamma=gamma)
                ocsvm.fit(X_train_sub)

                tr_scores = -ocsvm.decision_function(X_train)
                te_scores = -ocsvm.decision_function(X_test)

                min_len = 20 if anom_rate > 0.05 else 10
                lo_pct = max(85.0, 100.0 - anom_rate * 150)

                param_best_f1 = 0
                for pct in np.arange(lo_pct, 99.6, 1.0):
                    th = np.percentile(tr_scores, pct)
                    yp = (te_scores > th).astype(int)
                    yp = remove_short(yp, min_len)
                    r = eval_pred(yp, labels)
                    if r['f1'] > best_f1:
                        best_f1, best_pred = r['f1'], yp.copy()
                        best_nu, best_gamma = nu, gamma
                    if r['f1'] > param_best_f1:
                        param_best_f1 = r['f1']

                all_nu_gamma_results[(nu, gamma)] = param_best_f1

            except Exception as e:
                print(f"    nu={nu}, gamma={gamma:.4f} failed: {e}")

    r = eval_pred(best_pred, labels)
    print(f"  {ch:5s} [OCSVM] nu={best_nu:.2f}, gamma={best_gamma:.4f}: "
          f"P={r['p']:5.1%} R={r['r']:5.1%} F1={r['f1']:5.1%}")

    return {
        'channel': ch, 'best_nu': best_nu, 'best_gamma': best_gamma,
        'f1': r['f1'], 'precision': r['p'], 'recall': r['r'],
        'tp': r['tp'], 'fp': r['fp'], 'fn': r['fn'], 'tn': r['tn'],
        'anomaly_rate': anom_rate,
        'labels': labels, 'pred': best_pred, 'test_raw': test_arr,
        'all_nu_gamma_results': all_nu_gamma_results,
    }

print('OCSVM pipeline defined.')

---

## 8. Run All Channels

In [ ]:
channels = [ch for ch in EVAL_CHANNELS
            if ch in labels_df['chan_id'].values
            and os.path.exists(os.path.join(DATA_DIR, 'train', f'{ch}.npy'))
            and os.path.exists(os.path.join(DATA_DIR, 'test', f'{ch}.npy'))]

print(f'Channels: {len(channels)}')
print()

results = {}
for ch in channels:
    print(f'  Processing {ch}...')
    results[ch] = run_ocsvm_channel(ch, labels_df)
    print()

---

## 9. Aggregate Results

In [ ]:
tp = sum(r['tp'] for r in results.values())
fp = sum(r['fp'] for r in results.values())
fn = sum(r['fn'] for r in results.values())
tn = sum(r['tn'] for r in results.values())
micro_p = tp / (tp + fp) if (tp + fp) > 0 else 0
micro_r = tp / (tp + fn) if (tp + fn) > 0 else 0
micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0
macro_f1 = np.mean([r['f1'] for r in results.values()])

print('=' * 60)
print('  One-Class SVM (RBF) -- NASA SMAP/MSL')
print('=' * 60)
print(f'  Micro F1 : {micro_f1:.1%}  (P={micro_p:.1%}  R={micro_r:.1%})')
print(f'  Macro F1 : {macro_f1:.1%}')
print(f'  TP={tp}  FP={fp}  FN={fn}  TN={tn}')
print('=' * 60)

In [ ]:
per_ch = pd.DataFrame([{
    'Channel': r['channel'],
    'Anomaly %': f"{r['anomaly_rate']:.1%}",
    'nu': f"{r['best_nu']:.2f}",
    'gamma': f"{r['best_gamma']:.4f}",
    'F1': f"{r['f1']:.1%}",
    'Precision': f"{r['precision']:.1%}",
    'Recall': f"{r['recall']:.1%}",
    'TP': r['tp'], 'FP': r['fp'], 'FN': r['fn'],
} for r in results.values()])

per_ch

---

## 10. nu / gamma Sensitivity Analysis

**nu** controls the strictness of the normal boundary:
- Small nu: tight boundary (few outliers allowed, may miss some anomalies)
- Large nu: loose boundary (many outliers allowed, more false positives)

**gamma** controls the kernel bandwidth:
- Small gamma: smooth boundary (may underfit, miss local anomalies)
- Large gamma: complex boundary (may overfit, too many false positives)

The median heuristic provides a data-driven starting point for gamma.

In [ ]:
sample_ch = channels[0]
sens = results[sample_ch]['all_nu_gamma_results']

nu_vals = sorted(set(k[0] for k in sens.keys()))
gamma_vals = sorted(set(k[1] for k in sens.keys()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for nu in nu_vals:
    f1s = [sens.get((nu, g), 0) for g in gamma_vals]
    ax.plot(gamma_vals, f1s, marker='o', label=f'nu={nu:.2f}', linewidth=1.5)
ax.set_xlabel('gamma')
ax.set_ylabel('Best F1 Score')
ax.set_title(f'{sample_ch} -- F1 vs gamma (per nu)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
for gamma in gamma_vals:
    f1s = [sens.get((nu, gamma), 0) for nu in nu_vals]
    ax.plot(nu_vals, f1s, marker='s', label=f'gamma={gamma:.4f}', linewidth=1.5)
ax.set_xlabel('nu')
ax.set_ylabel('Best F1 Score')
ax.set_title(f'{sample_ch} -- F1 vs nu (per gamma)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle(f'OCSVM Parameter Sensitivity -- {sample_ch}', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, (len(channels) + 1) // 2, figsize=(18, 10))
axes = axes.flatten()

for i, ch in enumerate(channels):
    ax = axes[i]
    sens = results[ch]['all_nu_gamma_results']
    nu_vals = sorted(set(k[0] for k in sens.keys()))
    gamma_vals = sorted(set(k[1] for k in sens.keys()))

    f1_matrix = np.array([[sens.get((nu, g), 0) for g in gamma_vals] for nu in nu_vals])
    im = ax.imshow(f1_matrix, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_xticks(range(len(gamma_vals)))
    ax.set_xticklabels([f'{g:.3f}' for g in gamma_vals], fontsize=7, rotation=45)
    ax.set_yticks(range(len(nu_vals)))
    ax.set_yticklabels([f'{nu:.2f}' for nu in nu_vals], fontsize=8)
    ax.set_xlabel('gamma', fontsize=8)
    ax.set_ylabel('nu', fontsize=8)
    ax.set_title(f'{ch} (best F1={results[ch]["f1"]:.1%})', fontsize=9)

for j in range(len(channels), len(axes)):
    axes[j].set_visible(False)

fig.colorbar(im, ax=axes[:len(channels)], shrink=0.6, label='F1 Score')
plt.suptitle('OCSVM F1 Heatmap: nu (rows) x gamma (cols)', fontsize=13)
plt.tight_layout()
plt.show()

---

## 11. Anomaly Detection Visualizations

Red = ground truth anomaly, orange = false positive (predicted anomaly but normal).

In [ ]:
fig, axes = plt.subplots(len(channels), 1, figsize=(18, 3 * len(channels)))

for i, ch in enumerate(channels):
    r = results[ch]
    labels = r['labels']
    test_raw = r['test_raw']
    pred = r['pred']
    ax = axes[i]

    ax.plot(test_raw[:, 0], linewidth=0.3, color='steelblue', label='Telemetry')
    for s in range(len(labels)):
        if labels[s] == 1:
            ax.axvspan(s, s+1, alpha=0.3, color='red')
    for s in range(len(pred)):
        if pred[s] == 1 and labels[s] == 0:
            ax.axvspan(s, s+1, alpha=0.15, color='orange')

    ax.set_title(f'{ch}  [F1={r["f1"]:.1%}, nu={r["best_nu"]:.2f}, gamma={r["best_gamma"]:.4f}]  (anom {r["anomaly_rate"]:.1%})', fontsize=10)
    ax.set_ylabel('Value')

axes[-1].set_xlabel('Time index')
plt.suptitle('OCSVM Anomaly Detection: red=GT, orange=FP', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---

## 12. Comparison with All Other Methods

Loading results from previously run pipelines for comparison:
- **Isolation Forest (IF)**
- **LSTM-VAE + IF Ensemble**
- **Local Outlier Factor (LOF)**
- **One-Class SVM (OCSVM)**
- **LSTM-Forecast + LOF + IF Ensemble**

In [ ]:
if_results = {
    'P-1': 0.241, 'S-1': 0.568, 'E-1': 0.534, 'E-2': 0.239,
    'F-1': 0.000, 'G-1': 0.032, 'D-1': 0.761, 'M-5': 0.013,
}
lstm_vae_if_results = {
    'P-1': 0.164, 'S-1': 0.735, 'E-1': 0.644, 'E-2': 0.585,
    'F-1': 0.060, 'G-1': 0.131, 'D-1': 0.955, 'M-5': 0.753,
}
lof_results = {
    'P-1': 0.706, 'S-1': 0.938, 'E-1': 0.806, 'E-2': 0.340,
    'F-1': 0.707, 'G-1': 0.272, 'D-1': 0.945, 'M-5': 0.182,
}
ocsvm_results = {ch: results[ch]['f1'] for ch in channels}
lstm_lof_if_results = {
    'P-1': 0.696, 'S-1': 0.938, 'E-1': 0.806, 'E-2': 0.340,
    'F-1': 0.707, 'G-1': 0.272, 'D-1': 0.951, 'M-5': 0.902,
}

comparison = pd.DataFrame([{
    'Channel': ch,
    'Anomaly': f"{results[ch]['anomaly_rate']:.1%}",
    'IF F1': f"{if_results.get(ch, 0):.1%}",
    'LSTM-VAE+IF F1': f"{lstm_vae_if_results.get(ch, 0):.1%}",
    'LOF F1': f"{lof_results.get(ch, 0):.1%}",
    'OCSVM F1': f"{ocsvm_results.get(ch, 0):.1%}",
    'LSTM-LOF+IF F1': f"{lstm_lof_if_results.get(ch, 0):.1%}",
    'Best': max([('IF', if_results.get(ch, 0)),
                 ('LSTM-VAE+IF', lstm_vae_if_results.get(ch, 0)),
                 ('LOF', lof_results.get(ch, 0)),
                 ('OCSVM', ocsvm_results.get(ch, 0)),
                 ('LSTM-LOF+IF', lstm_lof_if_results.get(ch, 0))],
                key=lambda x: x[1])[0],
} for ch in channels])

comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(channels))
w = 0.16

if_f1 = [if_results.get(ch, 0) for ch in channels]
lstm_vae_f1 = [lstm_vae_if_results.get(ch, 0) for ch in channels]
lof_f1 = [lof_results.get(ch, 0) for ch in channels]
ocsvm_f1 = [ocsvm_results.get(ch, 0) for ch in channels]
lstm_lof_f1 = [lstm_lof_if_results.get(ch, 0) for ch in channels]

ax.bar(x - 2*w, if_f1, w, label='Isolation Forest', color='coral')
ax.bar(x - w, lstm_vae_f1, w, label='LSTM-VAE+IF', color='seagreen')
ax.bar(x, lof_f1, w, label='LOF', color='steelblue')
ax.bar(x + w, ocsvm_f1, w, label='OCSVM', color='darkorange')
ax.bar(x + 2*w, lstm_lof_f1, w, label='LSTM-LOF+IF', color='mediumpurple')

ax.set_xticks(x)
ax.set_xticklabels(channels)
ax.set_ylabel('F1 Score')
ax.set_title('Per-Channel F1: IF vs LSTM-VAE+IF vs LOF vs OCSVM vs LSTM-LOF+IF')
ax.legend(ncol=2, fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
macro_if = np.mean(list(if_results.values()))
macro_lstm_vae = np.mean(list(lstm_vae_if_results.values()))
macro_lof = np.mean(list(lof_results.values()))
macro_ocsvm = macro_f1
macro_lstm_lof = np.mean(list(lstm_lof_if_results.values()))

fig, ax = plt.subplots(figsize=(10, 4))
methods = ['Isolation Forest', 'LSTM-VAE+IF', 'LOF', 'OCSVM', 'LSTM-LOF+IF']
macro_f1s = [macro_if, macro_lstm_vae, macro_lof, macro_ocsvm, macro_lstm_lof]
colors = ['coral', 'seagreen', 'steelblue', 'darkorange', 'mediumpurple']

bars = ax.bar(methods, macro_f1s, color=colors, width=0.5)
for bar, val in zip(bars, macro_f1s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.1%}', ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('Macro F1 Score')
ax.set_title('Macro F1 Comparison Across All Methods')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 0.85)
plt.tight_layout()
plt.show()

---

## 13. Save Report

In [ ]:
summary = {
    'method': 'One-Class SVM (RBF)',
    'micro_f1': round(micro_f1, 4),
    'macro_f1': round(macro_f1, 4),
    'micro_p': round(micro_p, 4),
    'micro_r': round(micro_r, 4),
    'per_channel': {ch: {
        'f1': round(results[ch]['f1'], 4),
        'nu': round(results[ch]['best_nu'], 4),
        'gamma': round(results[ch]['best_gamma'], 6),
        'p': round(results[ch]['precision'], 4),
        'r': round(results[ch]['recall'], 4),
    } for ch in channels},
}
os.makedirs('reports', exist_ok=True)
with open('reports/ocsvm_metrics.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved to reports/ocsvm_metrics.json')